# GS Phase Recovery — Google Colab Demo

**Optical phase recovery from two dispersed intensity measurements.**

Based on: Solli, Gupta & Jalali, APL 95, 231108 (2009)  
Repo: https://github.com/ColinsCoding/Dispersion-Assisted-GS-Phase-Recovery

Run all cells top to bottom. No install needed beyond numpy/matplotlib (already in Colab).

In [ ]:
# Install if needed (Colab already has numpy/matplotlib)
# !pip install numpy matplotlib scipy  # uncomment if running locally
import numpy as np
import matplotlib.pyplot as plt
print('numpy:', np.__version__)
print('Ready.')

## Core algorithm (self-contained, no external dependencies)

In [ ]:
def disperse(E, D):
    """Apply GVD dispersion D to complex field E."""
    N  = len(E)
    nu = np.fft.fftfreq(N)
    H  = np.exp(1j * np.pi * D * nu**2)
    return np.fft.ifft(np.fft.fft(E) * H)

def recover_phase(I1, I2, D1, D2, n_iter=50, unit_amplitude=True):
    """Gerchberg-Saxton phase recovery from two intensity measurements."""
    I1 = np.maximum(np.asarray(I1, float), 0)
    I2 = np.maximum(np.asarray(I2, float), 0)
    N  = len(I1)
    rng = np.random.default_rng(0)
    E   = np.sqrt(I1) * np.exp(1j * rng.uniform(0, 2*np.pi, N))
    errors = []
    for i in range(n_iter):
        E1 = disperse(E, D1)
        E1 = np.exp(1j*np.angle(E1)) if unit_amplitude else np.sqrt(I1)*np.exp(1j*np.angle(E1))
        E  = disperse(E1, -D1)
        E2 = disperse(E, D2)
        E2 = np.exp(1j*np.angle(E2)) if unit_amplitude else np.sqrt(I2)*np.exp(1j*np.angle(E2))
        E  = disperse(E2, -D2)
        errors.append(float(np.mean(np.abs(np.abs(disperse(E,D1))**2 - I1))))
    return np.angle(E), errors

print('Algorithm loaded.')

## Generate synthetic QPSK data

In [ ]:
# Parameters
N      = 512
D1     = -5000.0
D2     = -5750.0
SNR_DB = 35

rng = np.random.default_rng(42)
n_sym   = N // 8
symbols = rng.choice([0,1,2,3], size=n_sym)
phases  = symbols * np.pi / 2
phi_true = np.repeat(phases, 8)[:N]
phi_true = np.convolve(phi_true, np.ones(4)/4, mode='same')

E_true = np.exp(1j * phi_true)
noise  = 10**(-SNR_DB/20)

I1 = np.maximum(np.abs(disperse(E_true, D1))**2 + noise*rng.standard_normal(N), 0)
I2 = np.maximum(np.abs(disperse(E_true, D2))**2 + noise*rng.standard_normal(N), 0)

print(f'I1 shape: {I1.shape}  range: [{I1.min():.3f}, {I1.max():.3f}]')
print(f'I2 shape: {I2.shape}  range: [{I2.min():.3f}, {I2.max():.3f}]')

plt.figure(figsize=(12,3))
plt.plot(I1, 'b', lw=0.8, label='I1 (D=-5000)')
plt.plot(I2, 'r', lw=0.8, alpha=0.7, label='I2 (D=-5750)')
plt.title('Input intensity measurements')
plt.xlabel('Sample'); plt.ylabel('Intensity'); plt.legend()
plt.tight_layout(); plt.show()

## Run GS recovery

In [ ]:
phi_hat, errors = recover_phase(I1, I2, D1, D2, n_iter=50)

# align global phase
offset  = np.angle(np.mean(np.exp(1j*(phi_true - phi_hat))))
phi_al  = phi_hat + offset
rms     = np.degrees(np.sqrt(np.mean((phi_true - phi_al)**2)))

print(f'Convergence: {errors[0]:.4f} -> {errors[-1]:.4f}')
print(f'RMS error vs ground truth: {rms:.2f} deg')

## Results

In [ ]:
t = np.arange(N)
fig, axes = plt.subplots(2, 2, figsize=(13, 8))
fig.suptitle('GS Phase Recovery Results', fontsize=14, fontweight='bold')

axes[0,0].plot(t, I1, 'b', lw=0.8, label='I1'); axes[0,0].plot(t, I2, 'r', lw=0.8, alpha=0.7, label='I2')
axes[0,0].set_title('Input intensities'); axes[0,0].legend()

axes[0,1].plot(t, np.degrees(phi_true), 'g--', lw=1.5, label='True')
axes[0,1].plot(t, np.degrees(phi_al),   'r',   lw=0.8, label=f'Recovered (RMS={rms:.1f} deg)')
axes[0,1].set_title('Phase: true vs recovered'); axes[0,1].legend()
axes[0,1].set_ylabel('Phase [deg]')

axes[1,0].semilogy(errors, 'k', lw=1.5)
axes[1,0].set_title('GS convergence'); axes[1,0].set_xlabel('Iteration'); axes[1,0].set_ylabel('Error')

axes[1,1].hist(np.degrees(phi_al), bins=60, color='steelblue', edgecolor='none')
axes[1,1].set_title('Recovered phase distribution'); axes[1,1].set_xlabel('Phase [deg]')

plt.tight_layout(); plt.show()

## Upload your own data

If you have real `I1.npy` / `I2.npy` files from a photonics lab:

In [ ]:
# Uncomment to upload files in Colab
# from google.colab import files
# uploaded = files.upload()   # select I1.npy and I2.npy
# I1_real = np.load('I1.npy').ravel().astype(float)
# I2_real = np.load('I2.npy').ravel().astype(float)
# D1_real = -695.0   # replace with your actual dispersion values
# D2_real = -800.0
# phi_real, errs_real = recover_phase(I1_real, I2_real, D1_real, D2_real)
# print('Recovered phase range:', np.degrees(phi_real.min()), 'to', np.degrees(phi_real.max()), 'deg')
print('Uncomment the block above and run to use real lab data.')

## What you just ran

```
Gerchberg-Saxton algorithm:

  Start: E = sqrt(I1) * exp(i * random_phase)
  Loop:
    E1 = disperse(E, D1)          # apply GVD
    E1 = exp(i * angle(E1))       # enforce I1 constraint  <- projection onto C1
    E  = disperse(E1, -D1)        # undo GVD
    E2 = disperse(E, D2)          # apply GVD2
    E2 = exp(i * angle(E2))       # enforce I2 constraint  <- projection onto C2
    E  = disperse(E2, -D2)        # undo GVD
  Return: angle(E)

H(nu) = exp(i*pi*D*nu^2)   <- dispersion transfer function
|H| = 1 everywhere          <- unitary, energy preserved (Parseval)
U(1) symmetry               <- global phase unobservable, need offset correction
```